# Cyclospora outbreak

## CDC Surveillance data

In [1]:
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
import pandas as pd
import re

In [2]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=True)

In [3]:
page = await browser.new_page()

In [4]:
await page.goto("https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html")

<Response url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' request=<Request url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' method='GET'>>

In [5]:
async with page.expect_download() as download_info:
    await page.click('a[aria-label="Download this data in a CSV file format."]')

download = await download_info.value
await download.save_as("data/data.csv")
print("Downloaded to:", await download.path())

Downloaded to: /tmp/playwright-artifacts-BEmNaF/ec0e0b91-e67e-4b0a-a19f-73193b21c1e7


In [6]:
df = pd.read_csv("data/data.csv")

In [7]:
df.head()

,Location,Number of Sick People
0,Alaska,1 to 10
1,Arizona,1 to 10
2,Arkansas,1 to 10
3,California,1 to 10
4,Colorado,1 to 10


In [8]:
df['min'] = df['Number of Sick People'].str.extract(r'(\d+)\s+to\s+\d+').astype(int)
df['max'] = df['Number of Sick People'].str.extract(r'\d+\s+to\s+(\d+)').astype(int)

In [9]:
df['average'] = df[['min', 'max']].mean(axis=1)

In [10]:
df.sort_values(by='average', ascending=False, inplace=True)
df.head()

,Location,Number of Sick People,min,max,average
16,Michigan,501 to 900,501,900,700.5
21,New York,161 to 300,161,300,230.5
22,North Carolina,81 to 160,81,160,120.5
20,New Jersey,31 to 80,31,80,55.5
28,Texas,31 to 80,31,80,55.5


In [11]:
df.to_csv("data/clean_data.csv", index=False)

## Cyclospora confirmed cases and notes

In [12]:
await page.goto("https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html")

<Response url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' request=<Request url='https://www.cdc.gov/cyclosporiasis/php/surveillance/index.html' method='GET'>>

In [13]:
fast_facts = await page.locator('div.dfe-section[data-section="cdc_generic_section_3"]').inner_text()
print(fast_facts)


2026 fast facts

As of July 13, 2026:

U.S. cases reported to CDC: 1,645
Hospitalizations: 141
Deaths: 0
States reporting cases: 34


In [14]:
fast_facts

'2026 fast facts\n\nAs of July 13, 2026:\n\nU.S. cases reported to CDC: 1,645\nHospitalizations: 141\nDeaths: 0\nStates reporting cases: 34'

In [15]:
labels = ['last_updated', 'total_cases', 'hospitalizations', 'deaths', 'states_reporting']

as_of_text = await page.locator('div.dfe-section[data-section="cdc_generic_section_3"] p').inner_text()

as_of_date = re.search(r'As of (.+?):', as_of_text).group(1).strip()

items = await page.locator('div.dfe-section[data-section="cdc_generic_section_3"] ul.nested-list li').all()

values = [as_of_date] + [((await item.inner_text()).split(':')[1].strip()) for item in items]

cdc_data = dict(zip(labels, values))
print(cdc_data)

{'last_updated': 'July 13, 2026', 'total_cases': '1,645', 'hospitalizations': '141', 'deaths': '0', 'states_reporting': '34'}


In [16]:
df_cdc = pd.DataFrame([cdc_data])

In [17]:
df_cdc.to_csv("data/cdc_notes.csv", index=False)

## Michigan data

In [18]:
await page.goto("https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks")

<Response url='https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks' request=<Request url='https://www.michigan.gov/mdhhs/keep-mi-healthy/infectious-diseases/infectious-disease-outbreaks' method='GET'>>

In [19]:
total_cases = await page.inner_text('p:has(strong:text("Total Cases:"))')
print(total_cases)


Total Cases: 3,762
As of July 9, 2026, 44 reported cases indicated they had been hospitalized.


In [20]:
match = re.search(r'Total Cases:\s*([\d,]+)', total_cases)
cases = match.group(1)  # "3,309"

# optional: convert to integer
michigan_cases = int(cases.replace(',', ''))  # 3309

print(cases)      # 3,309
print(michigan_cases)  # 3309

3,762
3762


In [21]:
last_updated = await page.inner_text('p:has(strong:text("Last updated:"))')
print(last_updated)

Last updated: July 15, 2026


## Combine data

In [22]:
df

,Location,Number of Sick People,min,max,average
16,Michigan,501 to 900,501,900,700.5
21,New York,161 to 300,161,300,230.5
22,North Carolina,81 to 160,81,160,120.5
20,New Jersey,31 to 80,31,80,55.5
28,Texas,31 to 80,31,80,55.5
8,Illinois,31 to 80,31,80,55.5
12,Kentucky,31 to 80,31,80,55.5
9,Indiana,31 to 80,31,80,55.5
30,Virginia,11 to 30,11,30,20.5
27,Tennessee,11 to 30,11,30,20.5


In [23]:
new_row = {
    'Location': 'Michigan',
    'Number of Sick People': michigan_cases,
    'min': michigan_cases,
    'max': michigan_cases,
    'average': michigan_cases
}

df_local = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

In [24]:
df_local

,Location,Number of Sick People,min,max,average
0,Michigan,501 to 900,501,900,700.5
1,New York,161 to 300,161,300,230.5
2,North Carolina,81 to 160,81,160,120.5
3,New Jersey,31 to 80,31,80,55.5
4,Texas,31 to 80,31,80,55.5
5,Illinois,31 to 80,31,80,55.5
6,Kentucky,31 to 80,31,80,55.5
7,Indiana,31 to 80,31,80,55.5
8,Virginia,11 to 30,11,30,20.5
9,Tennessee,11 to 30,11,30,20.5


In [25]:
df_local.to_csv("data/clean_data_local.csv", index=False)